# Lead-seizure sensitivity analysis

Addresses a standard rigor step in the seizure-prediction literature
(Truong et al. 2018; Winterhalder et al. 2003): **clustered seizures share a
preictal state and can inflate performance**, so the field restricts the positive
class to *lead* seizures — those isolated from any previous seizure by a minimum
interval (here **4 h**, configurable).

**Part 1** audits how much clustering exists in the analysed CHB-MIT cohort
(needs only the summary files — always runnable).

**Part 2** re-evaluates the flagship (PDC VAR(20) SVM, LOPO) using **lead
seizures only** and compares against the all-seizure result. It reuses the
existing `cache_pdc_var20/` features (so it needs `V6_PDC` to have run first) and
re-labels rather than recomputing, so the numbers are directly comparable to the
flagship. Deterministic seeds (no `hash()`), so it is reproducible on its own.


In [ ]:
# --- repo bootstrap (locate <repo>/src) ---
import sys as _sys, pathlib as _pl
REPO = _pl.Path.cwd()
while not (REPO / 'src' / 'config.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
_sys.path.insert(0, str(REPO / 'src'))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

from config import (DATA_ROOT, EXCLUDED_PATIENTS, STEP_SEC,
                    INTERICTAL_MULTIPLIER, MAX_INTERICTAL_ABS)
from summary_parser import parse_summary_with_times, parse_summary
from seizure_metrics import infer_seizure_groups

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

# ---- parameters ----
SEED          = 42
T_LEAD_HOURS  = 4.0                      # min gap for a seizure to count as 'lead'
T_LEAD        = T_LEAD_HOURS * 3600
PREICTAL_SEC  = 30 * 60                  # matches preprocessing (30-min preictal)
CAP_MULT      = INTERICTAL_MULTIPLIER    # 5
CAP_ABS       = MAX_INTERICTAL_ABS       # 5000
CACHE         = REPO / 'cache_pdc_var20'
OUT           = REPO / 'outputs'; OUT.mkdir(exist_ok=True)

COHORT = [f'chb{i:02d}' for i in range(1, 25) if f'chb{i:02d}' not in EXCLUDED_PATIENTS]
print(f'Cohort ({len(COHORT)}):', ' '.join(COHORT))
print(f'Lead criterion: >= {T_LEAD_HOURS} h since previous seizure offset')
np.random.seed(SEED)


## Part 1 — Lead-seizure audit (summary files only)

In [ ]:
def classify_seizures(pid):
    """Return every seizure for a patient, chronologically, tagged lead/clustered
    (by absolute inter-seizure gap) and reconstructable (>= 30 min into its file,
    matching the preprocessing rule that creates a preictal block)."""
    tl = parse_summary_with_times(str(Path(DATA_ROOT) / pid))
    seiz = []
    for info in tl.values():
        for (a_on, a_off), (r_on, r_off) in zip(info['seizures_abs'], info['seizures_rel']):
            seiz.append({'abs_on': a_on, 'abs_off': a_off, 'rel_on': r_on})
    seiz.sort(key=lambda s: s['abs_on'])
    prev_off = None
    for s in seiz:
        gap = np.inf if prev_off is None else s['abs_on'] - prev_off
        s['gap_h']           = gap / 3600
        s['lead']            = (prev_off is None) or (gap >= T_LEAD)
        s['reconstructable'] = s['rel_on'] >= PREICTAL_SEC
        prev_off = s['abs_off']
    return seiz

per_patient_seiz, rows = {}, []
for pid in COHORT:
    s = classify_seizures(pid); per_patient_seiz[pid] = s
    n_true = sum(len(v) for v in parse_summary(str(Path(DATA_ROOT) / pid)).values())
    if not s and n_true > 0:
        # summary has no 'File Start Time' (e.g. chb24) -> no absolute timeline,
        # so lead vs clustered cannot be determined. Flag, don't silently zero it.
        rows.append({'patient': pid, 'n_seizures': n_true, 'n_analysable': np.nan,
                     'n_lead': np.nan, 'n_clustered': np.nan, 'pct_clustered': np.nan,
                     'note': 'no clock times in summary - lead status undetermined'})
        continue
    rec  = [x for x in s if x['reconstructable']]              # seizures that enter the model
    nrec = len(rec); nlead = sum(x['lead'] for x in rec)
    rows.append({'patient': pid, 'n_seizures': len(s), 'n_analysable': nrec,
                 'n_lead': nlead, 'n_clustered': nrec - nlead,
                 'pct_clustered': round(100 * (nrec - nlead) / nrec, 1) if nrec else 0.0,
                 'note': ''})
audit = pd.DataFrame(rows)
audit.to_csv(OUT / 'lead_seizure_audit.csv', index=False)

valid   = audit.dropna(subset=['n_analysable'])
tot_rec = int(valid.n_analysable.sum()); tot_clu = int(valid.n_clustered.sum())
print(audit.to_string(index=False))
print(f'\nCohort (patients with a usable timeline): '
      f'{tot_clu}/{tot_rec} analysable seizures are clustered (non-lead) '
      f'= {100*tot_clu/tot_rec:.1f}%')
undet = audit[audit.note.str.len() > 0]
if len(undet):
    print('Undetermined (no clock times):',
          ', '.join(f"{r.patient}({int(r.n_seizures)} seiz)" for _, r in undet.iterrows()))
aff = valid[valid.n_clustered > 0].sort_values('pct_clustered', ascending=False)
print('Patients with clustered seizures:',
      ', '.join(f"{r.patient}({int(r.n_clustered)})" for _, r in aff.iterrows()) or 'none')


In [ ]:
# Bar chart: clustered vs lead per patient (timeline-available patients)
vplot = valid.reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(vplot))
ax.bar(x, vplot.n_lead, label='lead', color='#3a7bc8', edgecolor='white')
ax.bar(x, vplot.n_clustered, bottom=vplot.n_lead, label='clustered (non-lead)',
       color='#e8772e', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(vplot.patient, rotation=90, fontsize=8)
ax.set_ylabel('analysable seizures'); ax.legend()
ax.set_title(f'Lead vs clustered seizures per patient (>= {T_LEAD_HOURS} h criterion)')
plt.tight_layout(); plt.savefig(OUT / 'lead_seizure_audit.png', dpi=130); plt.show()
print('Saved outputs/lead_seizure_audit.png')


## Part 2 — Re-evaluate the flagship on lead seizures only

Reuses `cache_pdc_var20/` features. For each patient the contiguous preictal
blocks (`infer_seizure_groups`) are mapped, in chronological order, to the
reconstructable seizures from Part 1; the preictal windows of **clustered**
seizures are dropped, interictal is untouched. A per-patient **consistency check**
(block count == analysable-seizure count) guards against mis-alignment — patients
that fail it are left unchanged and reported, so no silent errors.
Everything is then capped (5x) and run through the same LOPO SVM (C=0.1).


In [ ]:
if not CACHE.exists():
    raise FileNotFoundError(
        f'{CACHE} not found. Run notebooks/01_experiments/V6_PDC.ipynb first '
        'to build the PDC feature cache, then re-run this notebook.')

def load_patient(pid):
    d = CACHE / pid
    return np.load(d / 'features.npy').astype(np.float32), np.load(d / 'labels.npy').astype(int)

DATA = {pid: load_patient(pid) for pid in COHORT if (CACHE / pid).exists()}
print(f'Loaded PDC cache for {len(DATA)} patients')

# Build the lead-only dataset by dropping clustered preictal blocks
DATA_lead, skipped, dropped = {}, [], {}
for pid, (X, y) in DATA.items():
    blocks = infer_seizure_groups(y)                       # chronological preictal blocks
    rec    = [s for s in per_patient_seiz[pid] if s['reconstructable']]
    if len(blocks) != len(rec):
        skipped.append((pid, len(blocks), len(rec)))
        DATA_lead[pid] = (X, y)                            # cannot map -> leave unchanged
        continue
    drop = set()
    for blk, s in zip(blocks, rec):
        if not s['lead']:
            drop.update(blk)
    if drop:
        keep = np.array([i for i in range(len(y)) if i not in drop])
        DATA_lead[pid] = (X[keep], y[keep]); dropped[pid] = len(drop)
    else:
        DATA_lead[pid] = (X, y)

print('Patients with preictal windows dropped (clustered):',
      {k: v for k, v in dropped.items()} or 'none')
if skipped:
    print('Block/seizure count mismatch (left unchanged, excluded from the effect):',
          [f'{p}: {b} blocks vs {r} seiz' for p, b, r in skipped])


In [ ]:
# Cap (5x, deterministic per-patient seed) + LOPO SVM  (same as the flagship)
def cap_all(D):
    out = {}
    for pid, (X, y) in D.items():
        pre = np.where(y == 1)[0]; ii = np.where(y == 0)[0]
        keep_n = min(len(ii), CAP_MULT * len(pre), CAP_ABS)
        if len(ii) > keep_n:
            r   = np.random.default_rng(SEED + int(pid[3:]))   # deterministic, no hash()
            sel = np.sort(np.concatenate([pre, r.choice(ii, keep_n, replace=False)]))
            out[pid] = (X[sel], y[sel])
        else:
            out[pid] = (X, y)
    return out

def make_model():
    return Pipeline([('scl', StandardScaler()),
                     ('clf', SVC(kernel='rbf', C=0.1, class_weight='balanced', random_state=SEED))])

def _score(m, X):
    return 1.0 / (1.0 + np.exp(-m.decision_function(X)))    # sigmoid-squashed SVM margin

def run_lopo(D):
    pids, rows = sorted(D), []
    for test in pids:
        Xtr = np.concatenate([D[p][0] for p in pids if p != test])
        ytr = np.concatenate([D[p][1] for p in pids if p != test])
        Xte, yte = D[test]
        if len(np.unique(yte)) < 2:
            continue
        m    = make_model().fit(Xtr, ytr)
        prob = _score(m, Xte); prev = float((yte == 1).mean())
        ap   = average_precision_score(yte, prob)
        rows.append({'patient': test, 'auc': roc_auc_score(yte, prob),
                     'auc_pr': ap, 'prevalence': prev,
                     'skill': (ap - prev) / (1 - prev)})
    return pd.DataFrame(rows)

R_all  = run_lopo(cap_all(DATA))
R_lead = run_lopo(cap_all(DATA_lead))
print('done')


In [ ]:
# Compare all-seizure vs lead-only
def summ(df): return df[['auc', 'auc_pr', 'skill']].mean()
s_all, s_lead = summ(R_all), summ(R_lead)
comp = pd.DataFrame({'all_seizures': s_all, 'lead_only': s_lead,
                     'delta': s_lead - s_all}).round(4)
comp.to_csv(OUT / 'lead_vs_all_lopo.csv')
print('Flagship PDC-SVM LOPO  —  all seizures vs lead-only:\n')
print(comp.to_string())
print(f'\nPatients affected (>=1 clustered preictal block dropped): {len(dropped)}')
d_aucpr = float(s_lead['auc_pr'] - s_all['auc_pr'])
print('\nInterpretation:')
if abs(d_aucpr) < 0.02:
    print(f'  AUC-PR changes by {d_aucpr:+.3f} — negligible: the reported performance is '
          'NOT an artifact of clustered seizures. State this as a robustness check.')
else:
    print(f'  AUC-PR changes by {d_aucpr:+.3f} — non-trivial: clustered seizures '
          f'{"inflated" if d_aucpr < 0 else "suppressed"} the headline; report the lead-only number.')
